# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning en un ambiente con espacio de estados más grande que FrozenLake y múltiples decisiones secuenciales.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.

**Respuesta:**

Taxi-v4 posee $|\mathcal{S}| = 500$ estados discretos y $|\mathcal{A}| = 6$ acciones discretas. El espacio de estados surge del producto cartesiano de las variables del sistema: $5 \text{ filas} \times 5 \text{ columnas} \times 5 \text{ ubicaciones del pasajero (4 paradas + taxi)} \times 4 \text{ destinos posibles} = 500$. A diferencia de FrozenLake (16 estados estrictamente posicionales), Taxi codifica conjuntamente la posición del vehículo y el estado interno de la tarea.


## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?

**Respuesta:**

$s_t \in \{0, \dots, 499\}$ representa la configuración del taxi y pasajero antes de actuar; $a_t \in \{0, \dots, 5\}$ es la acción ejecutada; $r_{t+1}$ es la recompensa escalar devuelta por el entorno ($-1$ por paso ordinario, $-10$ por recoger/dejar ilegalmente, $+20$ por entrega exitosa); y $s_{t+1}$ es el estado resultante tras la transición.


## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$

**Respuesta:**

$|\mathcal{S}| \times |\mathcal{A}| = 500 \times 6 = 3000$ valores $Q(s, a)$. Cada celda cuantifica el retorno acumulado esperado al tomar la acción $a$ en el estado $s$ y continuar bajo la política óptima.


## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    if np.random.rand() < epsilon:
        return env.action_space.sample()
    q_values = Q[state]
    max_q = np.max(q_values)
    best_actions = np.flatnonzero(q_values == max_q)
    return int(np.random.choice(best_actions))


## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    best_next_q = np.max(Q[next_state])
    td_target = reward + gamma * best_next_q
    td_error = td_target - Q[state, action]
    Q[state, action] += alpha * td_error
    return Q[state, action]


## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    """Entrena el agente Q-learning en el entorno de Taxi."""
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            # TODO 1: escoger acción con política epsilon-greedy
            action = choose_action(Q, state, epsilon, env)

            # TODO 2: ejecutar acción en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)

            # TODO 3: actualizar Q
            update_q(Q, state, action, reward, next_state, alpha, gamma)

            total_reward += reward
            state = next_state

            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards


## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?

**Respuesta:**

La recompensa media por episodio asciende desde valores fuertemente negativos (alrededor de $-800$ debido a penalizaciones de $-10$ y agotamiento del límite de 200 pasos) hasta estabilizarse en el rango de $+7.5$ a $+8.5$ hacia los episodios 1,500–2,000. La oscilación persistente en la fase estacionaria se debe a que $\epsilon = 0.10$ se mantiene fijo durante todo el entrenamiento, forzando un $10\%$ de acciones aleatorias. Sin embargo, la política determinista asociada $\arg\max_a Q(s,a)$ converge de forma consistente a la ruta óptima.


## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

**Respuesta:**

El agente no entrenado ejecuta un paseo aleatorio que agota los 200 pasos sin completar la entrega, colisionando reiteradamente y acumulando un retorno negativo cercano a $-200$. El agente entrenado navega de forma directa hacia la parada del pasajero, ejecuta la acción de recogida (`pickup`) con precisión y se desplaza por la trayectoria geodésica mínima hacia el destino, completando la tarea en 10–14 pasos con recompensa neta positiva ($+8$ a $+11$).


## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?

**Respuesta:**

Para el estado evaluado (123), la acción óptima corresponde al avance en la dirección que minimiza la distancia Manhattan hacia el pasajero. Un valor $Q$ mayor indica un retorno acumulado futuro superior. No puede interpretarse como la recompensa inmediata porque cualquier movimiento válido otorga exactamente el mismo costo $r = -1.0$; el valor $Q$ integra el retorno a largo plazo descontado vía la recursión de Bellman.


## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.

**Respuesta:**

Al modificar la tasa de aprendizaje a $\alpha = 0.3$ (frente a la tasa base $\alpha = 0.1$), el agente supera las recompensas negativas severas con mayor rapidez, estabilizándose hacia el episodio 800 en lugar del 1,500. Sin embargo, en el régimen estacionario exhibe mayor variabilidad debido a que cada acción exploratoria induce perturbaciones más marcadas en la tabla $Q$. Ambos regímenes convergen asintóticamente al mismo nivel de recompensa media ($+7.5$ a $+8.5$).


## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.


In [ ]:
# ============================================================
# EXPERIMENTO COMPARATIVO (Pregunta 7)
# Variación de alpha: alpha=0.3 vs alpha=0.1 (original)
# ============================================================
Q_exp = np.zeros((n_states, n_actions))
Q_exp_trained, rewards_exp = train_q_learning(
    env,
    Q_exp,
    episodes=5000,
    alpha=0.3,    # Variación: tasa de aprendizaje aumentada a 0.3
    gamma=0.95,
    epsilon=0.1
)

window = 100
ma_orig = np.convolve(rewards, np.ones(window)/window, mode="valid")
ma_exp = np.convolve(rewards_exp, np.ones(window)/window, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(ma_orig, label="Original (alpha=0.1)", alpha=0.8)
plt.plot(ma_exp, label="Experimento (alpha=0.3)", alpha=0.8)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio móvil (100 ep)")
plt.title("Comparación de hiperparámetros en Taxi-v4: alpha=0.1 vs alpha=0.3")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Recompensa media últimos 500 ep (alpha=0.1): {np.mean(rewards[-500:]):.2f}")
print(f"Recompensa media últimos 500 ep (alpha=0.3): {np.mean(rewards_exp[-500:]):.2f}")
